# Egyptian Auction Price Predictor
## Full Pipeline Notebook

**Model**: Ensemble of RF + LightGBM + XGBoost (log-target blend)  
**Features**: 100 TF-IDF (item_title + item_description) + 11 structured interaction features  
**Target**:  (EGP)

---

## 0. Setup

In [1]:
import sys, os
os.chdir(os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd())
sys.path.insert(0, os.getcwd())

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor

print("Libraries loaded.")

Libraries loaded.


## 1. Load & Explore Data

In [2]:
import pandas as pd

df_raw = pd.read_csv("../auction_dataset_egypt.csv")
print("Shape:", df_raw.shape)
print("\nColumns:", df_raw.columns.tolist())  # Fixed: Used \n for the newline
df_raw.head(3)

Shape: (20000, 18)

Columns: ['item_title', 'item_description', 'category', 'subcategory', 'brand', 'condition', 'product_age', 'starting_price', 'reserve_price', 'buy_now_price', 'auction_duration', 'listing_day_of_week', 'listing_hour', 'seller_rating', 'seller_total_sales', 'seller_account_age', 'verified_seller', 'final_selling_price']


,item_title,item_description,category,subcategory,brand,condition,product_age,starting_price,reserve_price,buy_now_price,auction_duration,listing_day_of_week,listing_hour,seller_rating,seller_total_sales,seller_account_age,verified_seller,final_selling_price
0,Casetify Impact Case iPhone 14 Pro Max Floral ...,"Casetify Impact case for iPhone 14 Pro Max, fl...",Mobile Accessories,Cases & Protection,Ringke,Like New,3,760,1810,2270,14,Saturday,21,4.3,10,3,0,1860
1,Apple MagSafe Charger 15W Original - Used 6 Mo...,Original Apple MagSafe Charger (MHXH3ZM/A) — 1...,Mobile Accessories,Chargers & Cables,Baseus,Very Good,31,720,910,1420,3,Saturday,21,4.1,22,20,0,1380
2,Orient Bambino Version VI Classic Automatic Wh...,Orient Bambino Classic Automatic V6 in white d...,Fashion,Watches,Casio,Very Good,35,1600,2830,3370,10,Thursday,19,4.8,36,35,1,3110


In [3]:
print("Price statistics (EGP):")
print(df_raw["final_selling_price"].describe().round(0))
# Fixed: Kept on one line and used \n for the line break
print("\nCategories:", sorted(df_raw["category"].unique().tolist())) 
print("Conditions:", sorted(df_raw["condition"].unique().tolist()))

Price statistics (EGP):
count      20000.0
mean       44160.0
std       123495.0
min           50.0
25%         1950.0
50%         9830.0
75%        26790.0
max      1401680.0
Name: final_selling_price, dtype: float64

Categories: ['Baby & Kids', 'Books & Education', 'Collectibles', 'Electronics', 'Fashion', 'Furniture & Home', 'Mobile Accessories', 'Sports & Fitness', 'Vehicles']
Conditions: ['Acceptable', 'Excellent', 'Good', 'Like New', 'New', 'Very Good']


In [4]:
# Sample titles and descriptions
for _, row in df_raw.head(4).iterrows():
    print(f"Title  : {row['item_title']}")
    print(f"Desc   : {row['item_description'][:100]}...")
    print(f"Price  : {row['final_selling_price']:,} EGP")
    print()

Title  : Casetify Impact Case iPhone 14 Pro Max Floral Design
Desc   : Casetify Impact case for iPhone 14 Pro Max, floral design. Double-layer protection, EcoShock technol...
Price  : 1,860 EGP

Title  : Apple MagSafe Charger 15W Original - Used 6 Months
Desc   : Original Apple MagSafe Charger (MHXH3ZM/A) — 1 meter cable. Purchased with iPhone 14 Pro. Used for 6...
Price  : 1,380 EGP

Title  : Orient Bambino Version VI Classic Automatic White FAC08004W0
Desc   : Orient Bambino Classic Automatic V6 in white dial with brown leather strap. Japanese automatic movem...
Price  : 3,110 EGP

Title  : Adidas Ultraboost 22 Core Black White Size 43
Desc   : Adidas Ultraboost 22 in Core Black / Cloud White, EU size 43. Purchased from Adidas Egypt (Citystars...
Price  : 7,500 EGP



## 2. Preprocessing

> **Key change**:  and  are NOT dropped here.
> They are converted to 100 TF-IDF features in Phase 4.

In [7]:
import os
import sys

# This forces Python to look in the root folder, where 'scripts' lives
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

# Now your imports will clear that red wiggly line and execute perfectly!
from scripts.preprocessing import (
    load_raw,
    step1_drop_duplicates,
    step2_drop_leakage,
    step3_keep_text,
    step4_fix_formatting,
    step5_handle_missing,
    step6_encode,
    step7_handle_outliers,
    step8_log_transform,
)
from scripts.utils import load_object, save_object

df = load_raw()
df = step1_drop_duplicates(df)
df = step2_drop_leakage(df)
df = step3_keep_text(df)         # keeps item_title, item_description
df = step4_fix_formatting(df)
df = step5_handle_missing(df)
df, encoders = step6_encode(df, fit=True)
df, caps     = step7_handle_outliers(df, fit=True)
df, log_cols = step8_log_transform(df)

save_object(encoders, "encoders.pkl")
save_object(caps,     "outlier_caps.pkl")
save_object(log_cols, "log_cols.pkl")
# Fixed: Kept on one line and used \n for the line break
print("\nShape after preprocessing:", df.shape)
print("Text cols still present:", [c for c in ["item_title", "item_description"] if c in df.columns])

ModuleNotFoundError: No module named 'scripts.preprocessing'

## 3. Feature Engineering

### Two types of features added:
1. **TF-IDF (100 features)** — from item_title + item_description  
2. **Structured interactions (10 features)** — e.g. seller_credibility, product_freshness

In [ ]:
from scripts.feature_engineering import engineer_features

df, vectorizer = engineer_features(df, fit=True)
print("Shape after feature engineering:", df.shape)
print("TF-IDF features:", len([c for c in df.columns if c.startswith("tfidf_")]))
print("Structured features:", len([c for c in df.columns if not c.startswith("tfidf_") and c != "final_selling_price"]))

## 4. Train/Test Split + Scaling

In [ ]:
TARGET = "final_selling_price"
X_all = df.drop(columns=[TARGET]); y_all = df[TARGET]

X_tr_r, X_te_r, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42)

scaler = RobustScaler()
X_tr_s = pd.DataFrame(scaler.fit_transform(X_tr_r), columns=X_tr_r.columns, index=X_tr_r.index)
X_te_s = pd.DataFrame(scaler.transform(X_te_r),     columns=X_te_r.columns, index=X_te_r.index)
save_object(scaler, "scaler.pkl")

print(f"Train: {X_tr_s.shape}   Test: {X_te_s.shape}")
print(f"y_train range: {y_train.min():,.0f} — {y_train.max():,.0f} EGP")

## 5. Feature Selection

In [ ]:
from scripts.feature_selection import select_features

Xtr, Xte, selected = select_features(X_tr_s, X_te_s, y_train)
print(f"
Final feature count: {len(selected)}")
print(f"  Structured : {len([c for c in selected if not c.startswith('tfidf_')])}")
print(f"  TF-IDF     : {len([c for c in selected if c.startswith('tfidf_')])}")
print(f"
Structured selected: {[c for c in selected if not c.startswith('tfidf_')]}")
print(f"
Top 10 TF-IDF selected: {[c for c in selected if c.startswith('tfidf_')][:10]}")

## 6. Model Training (Ensemble)

> For full Optuna tuning, run .
> Below uses strong fixed hyperparameters for notebook demonstration.

In [ ]:
log_y = np.log1p(y_train)
from scripts.utils import report_metrics
import numpy as np

lgbm = LGBMRegressor(n_estimators=1200, num_leaves=80, learning_rate=0.025,
    max_depth=8, subsample=0.75, colsample_bytree=0.65, reg_alpha=0.3,
    reg_lambda=0.5, min_child_samples=25, random_state=42, n_jobs=-1, verbose=-1)
lgbm.fit(Xtr, log_y)

xgb = XGBRegressor(n_estimators=900, max_depth=6, learning_rate=0.025,
    subsample=0.75, colsample_bytree=0.65, reg_alpha=0.3, reg_lambda=1.0,
    min_child_weight=8, random_state=42, n_jobs=-1, verbosity=0)
xgb.fit(Xtr, log_y)

rf = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=5,
    max_features=0.5, random_state=42, n_jobs=-1)
rf.fit(Xtr, log_y)

print("All 3 models trained.")

In [ ]:
def ensemble_predict(X):
    p = (rf.predict(X) + lgbm.predict(X) + xgb.predict(X)) / 3
    return np.maximum(np.expm1(p), 0)

print("=== FINAL RESULTS ===")
m_tr = report_metrics(y_train, ensemble_predict(Xtr), "Ensemble_Train")
m_te = report_metrics(y_test,  ensemble_predict(Xte), "Ensemble_Test")

pct = np.abs(y_test.values - ensemble_predict(Xte)) / y_test.values * 100
print(f"Median % error  : {np.median(pct):.1f}%")
print(f"Within 20%      : {(pct<=20).mean()*100:.1f}% of test samples")
print(f"Train-Test gap  : {m_tr['R2']-m_te['R2']:.4f}  (healthy = < 0.05)")

In [ ]:
# Save model bundle
bundle = {"rf": rf, "lgbm": lgbm, "xgb": xgb,
          "log_target": True, "type": "rf_lgbm_xgb_log_blend"}
save_object(bundle, "best_model.pkl")
save_object("RF_LGBM_XGB_Ensemble", "best_model_name.pkl")
print("Model bundle saved.")

## 7. Prediction Tests

Verify that each predicted price is **(a) logically correct** and **(b) inside the 80% interval**.

In [ ]:
from scripts.predict import predict_single_with_interval

tests = [
    {"label": "Anker USB-C Cable (budget)",
     "item_title": "Anker USB-C to USB-C 1m Braided Cable 240W",
     "item_description": "Anker 240W USB-C cable, 1 meter braided. Brand new sealed.",
     "category": "Mobile Accessories", "subcategory": "Chargers & Cables",
     "brand": "Anker", "condition": "New", "product_age": 0,
     "starting_price": 100, "auction_duration": 5,
     "listing_day_of_week": "Monday", "listing_hour": 10,
     "seller_rating": 4.2, "seller_total_sales": 30, "seller_account_age": 12, "verified_seller": 0},

    {"label": "MacBook Pro 14 M3 (like new)",
     "item_title": "Apple MacBook Pro 14-inch M3 Pro 18GB 512GB Space Black Like New",
     "item_description": "MacBook Pro 14 M3 Pro, 18GB, 512GB SSD. 12 months old, battery 94%. Original box and accessories.",
     "category": "Electronics", "subcategory": "Laptops",
     "brand": "Apple", "condition": "Like New", "product_age": 12,
     "starting_price": 25000, "auction_duration": 7,
     "listing_day_of_week": "Friday", "listing_hour": 19,
     "seller_rating": 4.9, "seller_total_sales": 120, "seller_account_age": 36, "verified_seller": 1},

    {"label": "Toyota Corolla 2019 (car)",
     "item_title": "Toyota Corolla 2019 1.6L Automatic 45000 km Pearl White Dealer",
     "item_description": "Toyota Corolla 2019, 1.6L automatic, 45000 km. One owner, full dealer service history. No accidents.",
     "category": "Vehicles", "subcategory": "Cars",
     "brand": "Generic", "condition": "Very Good", "product_age": 60,
     "starting_price": 200000, "auction_duration": 14,
     "listing_day_of_week": "Saturday", "listing_hour": 12,
     "seller_rating": 4.3, "seller_total_sales": 3, "seller_account_age": 6, "verified_seller": 0},

    {"label": "Samsung S24 Ultra (flagship)",
     "item_title": "Samsung Galaxy S24 Ultra 512GB Titanium Black Like New 3 Months",
     "item_description": "Samsung Galaxy S24 Ultra, 512GB. 3 months old. Original box, S-Pen, all accessories. Screen protector applied.",
     "category": "Electronics", "subcategory": "Smartphones",
     "brand": "Samsung", "condition": "Like New", "product_age": 3,
     "starting_price": 30000, "auction_duration": 5,
     "listing_day_of_week": "Friday", "listing_hour": 20,
     "seller_rating": 4.8, "seller_total_sales": 90, "seller_account_age": 30, "verified_seller": 1},

    {"label": "Adidas Ultraboost sneakers",
     "item_title": "Adidas Ultraboost 22 Core Black White Size 43 Excellent",
     "item_description": "Adidas Ultraboost 22, EU43. Worn 5 times, original box. Excellent condition, no yellowing.",
     "category": "Fashion", "subcategory": "Sneakers",
     "brand": "Adidas", "condition": "Excellent", "product_age": 6,
     "starting_price": 3000, "auction_duration": 5,
     "listing_day_of_week": "Wednesday", "listing_hour": 18,
     "seller_rating": 4.3, "seller_total_sales": 40, "seller_account_age": 18, "verified_seller": 0},
]

print(f"{'Item':<35} {'Predicted':>12} {'Low':>10} {'High':>12} {'In Range?':>10}")
print("-" * 82)
for t in tests:
    label = t.pop("label")
    r = predict_single_with_interval(t)
    p, lo, hi = r["predicted_price"], r["price_range_low"], r["price_range_high"]
    ok = "✓" if lo <= p <= hi else "✗ FAIL"
    print(f"{label:<35} {p:>12,.0f} {lo:>10,.0f} {hi:>12,.0f} {ok:>10}")

## 8. Sample API Call

Once the FastAPI server is running ():

In [ ]:
sample_payload = {
    "item_title": "Apple iPhone 15 Pro Max 256GB Natural Titanium Like New",
    "item_description": (
        "iPhone 15 Pro Max, 256GB, Natural Titanium. Used 6 months. "
        "Battery health 97%. Original box and all accessories included. "
        "MagSafe case included. No scratches."
    ),
    "category": "Electronics",
    "subcategory": "Smartphones",
    "brand": "Apple",
    "condition": "Like New",
    "product_age": 6,
    "starting_price": 35000,
    "auction_duration": 7,
    "listing_day_of_week": "Saturday",
    "listing_hour": 20,
    "seller_rating": 4.9,
    "seller_total_sales": 80,
    "seller_account_age": 24,
    "verified_seller": 1,
}

result = predict_single_with_interval(sample_payload)
print(f"Item       : {sample_payload['item_title']}")
print(f"Predicted  : {result['predicted_price']:>12,.0f} EGP")
print(f"Range Low  : {result['price_range_low']:>12,.0f} EGP  ← set as reserve price")
print(f"Range High : {result['price_range_high']:>12,.0f} EGP  ← set as buy-now price")